# 04 — Bayesian integration of RNA and ATAC evidence

**Objective.** Compute a cell-by-module RNA prior, an ATAC log-likelihood, and a posterior-like module score using the public PBMC inputs and the responsibility generated in notebook 03.

**Outputs.** `prior.parquet`, `log_likelihood.parquet`, and `posterior.parquet` under `WORK_ROOT/04_bayesian`.

## Configuration and imports

In [ ]:
import gc
import os
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import mudata
import numpy as np
import pandas as pd
from scipy import sparse

from _tutorial_utils import (
    build_quadrant_summary,
    compare_dataframe_checkpoint,
    ensure_crane_grn_importable,
    lineage_match_summary,
    plot_heatmap,
    require_path,
)

ensure_crane_grn_importable()
from crane_grn.io import (
    extract_cistopic_matrices,
    extract_factor_expression,
    get_modality,
    normalize_log1p_expression,
)
from crane_grn.likelihood import compute_log_likelihood
from crane_grn.posterior import compute_posterior
from crane_grn.prior import compute_module_prior

DATA_ROOT = Path(os.environ.get("CRANE_GRN_DATA_ROOT", "../../CRANE_GRN_data")).expanduser().resolve()
WORK_ROOT = Path(os.environ.get("CRANE_GRN_TUTORIAL_WORK", "./results")).expanduser().resolve()
PBMC_INPUT_ROOT = DATA_ROOT / "PBMC" / "prepared_inputs"
PBMC_REFERENCE_ROOT = DATA_ROOT / "PBMC" / "outputs"
RESPONSIBILITY_ROOT = WORK_ROOT / "03_responsibility"
OUTPUT_ROOT = WORK_ROOT / "04_bayesian"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## RNA-informed prior

The public `scRNA_counts` modality is the raw count representation. Normalization is performed over the complete gene library to a total of 10,000 counts per cell, followed by `log1p`; only then are the 17 module factors materialized densely. Passing an already normalized matrix here would incorrectly normalize it a second time.

In [ ]:
scplus = mudata.read(str(require_path(PBMC_INPUT_ROOT / "scplusmdata.h5mu", "PBMC scplusmdata input")))
rna = get_modality(scplus, "scRNA_counts")
atac = get_modality(scplus, "scATAC_counts")
factor_by_module = pd.read_parquet(require_path(RESPONSIBILITY_ROOT / "factor_by_module.parquet", "tutorial factor-by-module matrix"))
responsibility_hvr = pd.read_parquet(require_path(RESPONSIBILITY_ROOT / "responsibility_hvr.parquet", "tutorial HVR responsibility"))

raw_values = rna.X.data if sparse.issparse(rna.X) else np.asarray(rna.X).ravel()
if raw_values.size == 0 or np.nanmin(raw_values) < 0 or np.nanmax(raw_values) <= 1:
    raise ValueError("scRNA_counts does not look like a non-negative raw count matrix.")
normalized_rna = normalize_log1p_expression(rna.X, target_sum=10000.0, dtype=np.float32)
factor_expression = extract_factor_expression(
    normalized_expression=normalized_rna,
    cell_names=rna.obs_names,
    gene_names=rna.var_names,
    factors=factor_by_module.index,
    dtype=np.float32,
)

if "celltype" not in rna.obs:
    raise KeyError("Expected group column 'celltype' was not found in scRNA_counts.obs.")
rna_groups = rna.obs["celltype"].astype(str).copy()
prior, prior_module_diagnostics, prior_gene_diagnostics, prior_tier_diagnostics = (
    compute_module_prior(
        normalized_expression=normalized_rna,
        cell_names=rna.obs_names,
        gene_names=rna.var_names,
        factor_by_module=factor_by_module,
        groups=rna_groups,
        prior_clip_max=0.95,
        beta_slope=0.5,
        z_min_scale=0.05,
        z_cap=1.5,
        tier_min_separation=0.35,
        prior_clip_min=0.01,
        minimum_cells_per_group=100,
        input_is_log1p=True,
        return_diagnostics=True,
    )
)
prior.to_parquet(OUTPUT_ROOT / "prior.parquet")
prior_module_diagnostics.to_csv(OUTPUT_ROOT / "prior_module_diagnostics.tsv", sep="\t", index=False)
print("Prior:", prior.shape)

## ATAC-informed log-likelihood

The selected cisTopic model reconstructs accessibility from region-topic loadings and cell-topic activities. Observed ATAC accessibility contributes a responsibility-weighted Bernoulli log-likelihood. 

In [ ]:
with require_path(PBMC_INPUT_ROOT / "cistopic_obj.pkl", "PBMC cisTopic input").open("rb") as handle:
    cistopic = pickle.load(handle)
region_by_topic, cell_by_topic = extract_cistopic_matrices(cistopic)
if "cell_type" not in atac.obs:
    raise KeyError("Expected group column 'cell_type' was not found in scATAC_counts.obs.")
atac_groups = atac.obs["cell_type"].astype(str).copy()
log_likelihood = compute_log_likelihood(
    responsibility=responsibility_hvr,
    region_by_topic=region_by_topic,
    cell_by_topic=cell_by_topic,
    observed_accessibility=atac,
    groups=atac_groups,
    responsibility_exponent=1.5,
    batch_size=256,
    epsilon=1e-6,
    dtype="float32",
    normalization_cell_fraction=0.5,
    normalization_region_fraction=0.5,
    minimum_coverage=10,
    minimum_cells_per_group=50,
)
log_likelihood.to_parquet(OUTPUT_ROOT / "log_likelihood.parquet")
finite_likelihood = log_likelihood.to_numpy(dtype=float, copy=False)
finite_likelihood = finite_likelihood[np.isfinite(finite_likelihood)]
if finite_likelihood.size == 0 or finite_likelihood.max() > 1e-6:
    raise ValueError("Unexpected PBMC log-likelihood range.")
print("Log-likelihood:", log_likelihood.shape)
print("Finite quantiles:", np.quantile(finite_likelihood, [0.01, 0.5, 0.99]))
print("Display-only likelihood-scale quantiles:", np.quantile(np.exp(finite_likelihood), [0.01, 0.5, 0.99]))

## Posterior integration

Modules lacking sufficient likelihood coverage are removed during posterior integration. The expected reduction from 890 to 530 modules is therefore a coverage result, not an error. Posterior values are cell/module scores; they are not a categorical distribution and do not sum to one over modules.

In [ ]:
posterior = compute_posterior(
    log_likelihood=log_likelihood,
    prior=prior,
    groups=atac_groups,
    prior_weight=1.0,
    evidence_weight=1.0,
    prior_clip=(1e-6, 1.0 - 1e-6),
    logit_cap=None,
)
posterior.to_parquet(OUTPUT_ROOT / "posterior.parquet")
print("Posterior:", posterior.shape)
print("Posterior row-sum quantiles (not constrained to one):", posterior.sum(axis=1).quantile([0.01, 0.5, 0.99]).to_dict())

## Compact manuscript-relevant validation

The first view is a prior/evidence quadrant summary. The second is a descriptive lineage consistency summary.

In [ ]:
quadrant_pairs, quadrant_summary, quadrant_thresholds = (
    build_quadrant_summary(
        prior=prior,
        log_likelihood=log_likelihood,
        posterior=posterior,
        groups=atac_groups,
        sample_points=10000,
        seed=0,
        heatmap_cells=3000,
    )
)

fig, ax = plt.subplots(figsize=(6.5, 5.5))

points = ax.scatter(
    quadrant_pairs["prior"],
    quadrant_pairs["evidence"],
    c=quadrant_pairs["posterior"],
    cmap="viridis",
    s=10,
    alpha=0.55,
    linewidths=0,
    rasterized=True,
)

# Manuscript-matched low/high quadrant thresholds.
ax.axvline(
    quadrant_thresholds["prior_low"],
    color="#777777",
    linestyle="--",
    linewidth=0.9,
)
ax.axvline(
    quadrant_thresholds["prior_high"],
    color="#777777",
    linestyle="--",
    linewidth=0.9,
)
ax.axhline(
    quadrant_thresholds["evidence_low"],
    color="#777777",
    linestyle="--",
    linewidth=0.9,
)
ax.axhline(
    quadrant_thresholds["evidence_high"],
    color="#777777",
    linestyle="--",
    linewidth=0.9,
)

ax.set(
    xlabel="RNA-based prior probability",
    ylabel="Standardized ATAC evidence",
    title="PBMC prior–evidence diagnostic",
)

colorbar = fig.colorbar(points, ax=ax)
colorbar.set_label("Posterior probability")

fig.tight_layout()
plt.show()

threshold_table = pd.DataFrame(
    [
        {
            "prior_low": quadrant_thresholds["prior_low"],
            "prior_high": quadrant_thresholds["prior_high"],
            "likelihood_low": quadrant_thresholds["evidence_low"],
            "likelihood_high": quadrant_thresholds["evidence_high"],
            "sampled_pairs": quadrant_thresholds["n_sampled_pairs"],
            "non_saturated_pairs": (
                quadrant_thresholds["n_non_saturated_pairs"]
            ),
        }
    ]
)

display(threshold_table)
display(quadrant_summary)

In [ ]:
lineage_summary = pd.concat(
    [
        lineage_match_summary(prior.loc[:, posterior.columns], atac_groups, "Prior", top_quantile=0.75),
        lineage_match_summary(log_likelihood.loc[:, posterior.columns], atac_groups, "ATAC evidence", top_quantile=0.75),
        lineage_match_summary(posterior, atac_groups, "Posterior", top_quantile=0.75),
    ],
    ignore_index=True,
)
lineage_evaluable = lineage_summary.dropna(subset=["expected_match"]).copy()
lineage_evaluable["expected_match"] = lineage_evaluable["expected_match"].astype(float)
match_matrix = lineage_evaluable.pivot_table(
    index="lineage", columns="score", values="expected_match", aggfunc="mean"
).reindex(index=["T/NK", "B", "Myeloid"], columns=["Prior", "ATAC evidence", "Posterior"])
plot_heatmap(match_matrix, "Lineage-consistent peak-group fraction", colorbar_label="fraction", cmap="YlGnBu")
plt.show()
display(match_matrix)

## Interpretation, outputs, and next step

RNA supplies factor-state information, while the ATAC term evaluates responsibility-weighted accessibility in the cisTopic reconstruction. Their integration retains 530 covered modules.

**Next step:** run `05_support.py` in the main CRANE environment to connect posterior-gated module activity to direct eRegulons.